In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ANSI_RE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
NUM = r"([0-9]+(?:\.[0-9]+)?(?:e-?\d+)?)"


def extract_yolo_training_rows(notebook_path):
    """Doc output notebook Ultralytics va tra ve DataFrame theo epoch."""
    notebook_path = Path(notebook_path)
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))

    text_parts = []
    for cell in nb.get("cells", []):
        for output in cell.get("outputs", []):
            if "text" in output:
                text_parts.append("".join(output["text"]))

    text = ANSI_RE.sub("", "\n".join(text_parts)).replace("\r", "\n")

    rows = []
    pending_loss = None

    for line in text.splitlines():
        clean = " ".join(line.split())

        loss_match = re.match(
            rf"^(\d{{1,3}})/100\s+\S+\s+{NUM}\s+{NUM}\s+{NUM}\s+",
            clean,
        )
        if loss_match:
            pending_loss = {
                "epoch": int(loss_match.group(1)),
                "box_loss": float(loss_match.group(2)),
                "cls_loss": float(loss_match.group(3)),
                "dfl_loss": float(loss_match.group(4)),
            }
            continue

        metric_match = re.match(
            rf"^all\s+(\d+)\s+(\d+)\s+{NUM}\s+{NUM}\s+{NUM}\s+{NUM}",
            clean,
        )
        if metric_match and pending_loss:
            rows.append(
                {
                    **pending_loss,
                    "images": int(metric_match.group(1)),
                    "instances": int(metric_match.group(2)),
                    "precision": float(metric_match.group(3)),
                    "recall": float(metric_match.group(4)),
                    "map50": float(metric_match.group(5)),
                    "map50_95": float(metric_match.group(6)),
                }
            )
            pending_loss = None

    df = pd.DataFrame(rows)
    if len(df) != 100:
        print(f"Canh bao: {notebook_path} chi trich duoc {len(df)} epoch.")
    return df


yolo8_df = extract_yolo_training_rows("train files/yolo8s_focalloss.ipynb")
yolo11_df = extract_yolo_training_rows("train files/yolo11s_focalloss.ipynb")

# Neu can mang Python rieng:
yolo8_epochs = yolo8_df["epoch"].to_list()
yolo8_box_loss = yolo8_df["box_loss"].to_list()
yolo8_cls_loss = yolo8_df["cls_loss"].to_list()
yolo8_dfl_loss = yolo8_df["dfl_loss"].to_list()
yolo8_precision = yolo8_df["precision"].to_list()
yolo8_recall = yolo8_df["recall"].to_list()
yolo8_map50 = yolo8_df["map50"].to_list()
yolo8_map50_95 = yolo8_df["map50_95"].to_list()

yolo11_epochs = yolo11_df["epoch"].to_list()
yolo11_box_loss = yolo11_df["box_loss"].to_list()
yolo11_cls_loss = yolo11_df["cls_loss"].to_list()
yolo11_dfl_loss = yolo11_df["dfl_loss"].to_list()
yolo11_precision = yolo11_df["precision"].to_list()
yolo11_recall = yolo11_df["recall"].to_list()
yolo11_map50 = yolo11_df["map50"].to_list()
yolo11_map50_95 = yolo11_df["map50_95"].to_list()